# Regularization Strength Tuning & Solver Architecture Lab

Selecting the optimal regularization strength requires evaluating performance across logarithmic decades. Furthermore, scikit-learn's optimization solvers have strict mathematical compatibility rules with specific penalties (e.g. L1 vs L2) and multiclass formulations. This lab tunes Ridge and Lasso using `GridSearchCV`, benchmarks specialized warm-start path algorithms (`RidgeCV`, `LassoCV`), and verifies solver compatibility.

In [ ]:
import numpy as np
from sklearn.linear_model import (
    Ridge, Lasso, RidgeCV, LassoCV, LogisticRegression
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.datasets import make_regression, make_classification

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Tuning Ridge Regularization via GridSearchCV

Sweep $\alpha \in [0.001, 100.0]$ on a standardized regression pipeline using 5-fold cross-validation.

In [ ]:
X, y = make_regression(n_samples=100, n_features=10, noise=10.0, random_state=42)

pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
grid = GridSearchCV(
    pipe_ridge,
    {'model__alpha': alphas},
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_mean_squared_error'
).fit(X, y)

print(f"{'Alpha':<10} {'Mean CV MSE':<18} {'Std Dev':<14}")
print("-" * 42)
for a, score, std in zip(alphas, -grid.cv_results_['mean_test_score'], grid.cv_results_['std_test_score']):
    marker = " (Optimal)" if a == grid.best_params_['model__alpha'] else ""
    print(f"{a:<10.3f} {score:<18.4f} {std:<14.4f}{marker}")

## 2. Ultra-Fast Path Solvers: RidgeCV & LassoCV

Instead of training redundant models in a nested loop, specialized path algorithms (`RidgeCV` with closed-form LOO-CV and `LassoCV` with coordinate descent warm-starts) compute the entire regularization path in a fraction of the time.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

# RidgeCV uses efficient closed-form Leave-One-Out CV for free
ridge_cv = RidgeCV(alphas=alphas, store_cv_values=True).fit(X_scaled, y)
print(f"RidgeCV Best Alpha: {ridge_cv.alpha_:.3f}")

# LassoCV traces the coordinate descent path
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_scaled, y)
print(f"LassoCV Best Alpha: {lasso_cv.alpha_:.3f}")
print(f"Non-zero features selected: {np.sum(lasso_cv.coef_ != 0)} / {len(lasso_cv.coef_)}")

## 3. Testing Scikit-Learn Solver Compatibility

Demonstrate how unsupported solver-penalty pairings fail with explicit `ValueError` exceptions.

In [ ]:
X_c, y_c = make_classification(n_samples=50, n_features=5, n_classes=3, n_informative=3, random_state=42)

test_configs = [
    ('lbfgs', 'l2', True),
    ('lbfgs', 'l1', False),        # Incompatible: L-BFGS cannot handle non-differentiable L1
    ('liblinear', 'l1', False),    # Incompatible with multinomial
    ('saga', 'l1', True),         # Universal support
    ('saga', 'elasticnet', True)   # Universal support
]

print(f"{'Solver':<12} {'Penalty':<14} {'Expected':<12} {'Actual Execution'}")
print("-" * 56)
for solver, penalty, expected in test_configs:
    try:
        clf = LogisticRegression(solver=solver, penalty=penalty, l1_ratio=0.5 if penalty == 'elasticnet' else None, max_iter=200)
        clf.fit(X_c, y_c)
        res = "SUCCESS"
    except ValueError as e:
        res = f"FAILED: {str(e)[:25]}..."
    print(f"{solver:<12} {penalty:<14} {str(expected):<12} {res}")